# SQL LIKE Search — Recruitment Matcher

> **Mục đích notebook này:** Minh hoạ cách tìm kiếm ứng viên bằng **SQL LIKE (keyword cứng)**  
> sử dụng **cùng dataset** với pipeline Milvus, để so sánh trực tiếp kết quả và chỉ ra  
> lý do tại sao **Vector Search (Milvus) vượt trội** trong bài toán tuyển dụng ngữ nghĩa.

---

## Pipeline
```
HuggingFace Dataset (500 JD + 300 CV)
         ↓
   Clean & normalize
         ↓
   SQLite in-memory DB
         ↓
  SQL LIKE keyword search
         ↓
  So sánh với Milvus results
```

**Thứ tự chạy:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7

In [1]:
# ============================================================
# CELL 1: Cài đặt thư viện cần thiết
# ============================================================
# Chỉ cần datasets (để load HuggingFace) + tabulate (để hiển thị đẹp)
# SQLite đã có sẵn trong Python stdlib — KHÔNG cần cài thêm
!pip install datasets tabulate --quiet

print("✅ Thư viện đã sẵn sàng")
print("   - datasets  : load HuggingFace dataset")
print("   - sqlite3   : built-in Python (không cần cài)")
print("   - tabulate  : hiển thị bảng kết quả đẹp")

✅ Thư viện đã sẵn sàng
   - datasets  : load HuggingFace dataset
   - sqlite3   : built-in Python (không cần cài)
   - tabulate  : hiển thị bảng kết quả đẹp


In [2]:
# ============================================================
# CELL 2: Load dataset từ HuggingFace
# (Giống hoàn toàn Cell 1 trong notebook Milvus)
# ============================================================
from datasets import load_dataset

jd_data = load_dataset(
    "lang-uk/recruitment-dataset-job-descriptions-english",
    split="train[:500]"
)
cv_data = load_dataset(
    "lang-uk/recruitment-dataset-candidate-profiles-english",
    split="train[:300]"
)

print(f"JD columns : {jd_data.column_names}")
print(f"CV columns : {cv_data.column_names}")
print(f"Loaded     : {len(jd_data)} JDs, {len(cv_data)} CVs")

c:\Users\Tien Cong\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JD columns : ['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', '__index_level_0__']
CV columns : ['Position', 'Moreinfo', 'Looking For', 'Highlights', 'Primary Keyword', 'English Level', 'Experience Years', 'CV', 'CV_lang', 'id', '__index_level_0__']
Loaded     : 500 JDs, 300 CVs


In [3]:
# ============================================================
# CELL 3: Clean data
# (Giống Cell 2 Milvus — nhưng KHÔNG cần embed, KHÔNG cần vector)
# ============================================================
import re

def clean_text(text):
    """Xử lý null, khoảng trắng thừa, ký tự đặc biệt"""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def truncate_text(text, max_length=500):
    """Cắt tại ranh giới từ"""
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] if last_space > 0 else truncated

# --- Clean JD ---
cleaned_jd = []
for i, item in enumerate(jd_data):
    description = clean_text(item.get("Long Description"))
    if len(description) < 20:
        continue
    cleaned_jd.append({
        "id":          i + 1,
        "position":    clean_text(item.get("Position")) or "Unknown Position",
        "description": truncate_text(description, 2000),
        "company":     clean_text(item.get("Company Name")) or "Unknown Company",
        "keyword":     clean_text(item.get("Primary Keyword")) or "",
        "exp_years":   clean_text(str(item.get("Exp Years") or "")),
    })

# --- Clean CV ---
cleaned_cv = []
for i, item in enumerate(cv_data):
    cv_text = clean_text(item.get("CV"))
    if len(cv_text) < 20:
        continue
    cleaned_cv.append({
        "id":           i + 1,
        "position":     clean_text(item.get("Position")) or "Unknown Position",
        "cv_text":      truncate_text(cv_text, 2000),
        "highlights":   clean_text(item.get("Highlights")) or "",
        "keyword":      clean_text(item.get("Primary Keyword")) or "",
        "exp_years":    clean_text(str(item.get("Experience Years") or "")),
        "looking_for":  clean_text(item.get("Looking For")) or "",
    })

print(f"JD sạch: {len(cleaned_jd)} bản ghi")
print(f"CV sạch: {len(cleaned_cv)} bản ghi")
print("\n📝 NOTE: Bước này KHÔNG tạo vector embedding")
print("   SQL chỉ lưu text thô — đây là điểm khác biệt căn bản với Milvus")

JD sạch: 500 bản ghi
CV sạch: 300 bản ghi

📝 NOTE: Bước này KHÔNG tạo vector embedding
   SQL chỉ lưu text thô — đây là điểm khác biệt căn bản với Milvus


In [4]:
# ============================================================
# CELL 4: Tạo SQLite database và insert data
# (Thay thế Cell 4+5 Milvus — dùng SQL thay vì Vector DB)
# ============================================================
import sqlite3

# Dùng in-memory database để demo
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Truy cập kết quả theo tên cột
cur = conn.cursor()

# --- Tạo bảng job_descriptions ---
cur.execute("""
    CREATE TABLE job_descriptions (
        id          INTEGER PRIMARY KEY,
        position    TEXT,
        description TEXT,
        company     TEXT,
        keyword     TEXT,
        exp_years   TEXT
    )
""")

# --- Tạo bảng cvs ---
cur.execute("""
    CREATE TABLE cvs (
        id          INTEGER PRIMARY KEY,
        position    TEXT,
        cv_text     TEXT,
        highlights  TEXT,
        keyword     TEXT,
        exp_years   TEXT,
        looking_for TEXT
    )
""")

# --- Insert JDs ---
cur.executemany("""
    INSERT INTO job_descriptions
        (id, position, description, company, keyword, exp_years)
    VALUES
        (:id, :position, :description, :company, :keyword, :exp_years)
""", cleaned_jd)

# --- Insert CVs ---
cur.executemany("""
    INSERT INTO cvs
        (id, position, cv_text, highlights, keyword, exp_years, looking_for)
    VALUES
        (:id, :position, :cv_text, :highlights, :keyword, :exp_years, :looking_for)
""", cleaned_cv)

conn.commit()

# Tạo index trên các cột text để tăng tốc LIKE (thực tế không giúp nhiều với LIKE %...%)
cur.execute("CREATE INDEX idx_cv_position ON cvs(position)")
cur.execute("CREATE INDEX idx_cv_keyword  ON cvs(keyword)")
cur.execute("CREATE INDEX idx_jd_position ON job_descriptions(position)")

print(f"✅ SQLite DB tạo xong (in-memory)")
print(f"   Bảng job_descriptions : {len(cleaned_jd)} hàng")
print(f"   Bảng cvs              : {len(cleaned_cv)} hàng")
print("\n⚠️  Index trên TEXT không giúp ích với LIKE '%keyword%'")
print("   SQLite phải quét toàn bộ bảng (Full Table Scan) cho mỗi query")

✅ SQLite DB tạo xong (in-memory)
   Bảng job_descriptions : 500 hàng
   Bảng cvs              : 300 hàng

⚠️  Index trên TEXT không giúp ích với LIKE '%keyword%'
   SQLite phải quét toàn bộ bảng (Full Table Scan) cho mỗi query


In [5]:
# ============================================================
# CELL 5: Định nghĩa hàm SQL LIKE Search
# ============================================================
import time
from tabulate import tabulate

def extract_keywords(query: str) -> list[str]:
    """
    Trích xuất từ khoá từ câu query tự nhiên.
    → Bỏ stop words, lấy các từ có nghĩa (>= 3 ký tự)
    
    ĐÂY LÀ ĐIỂM YẾU CỐT LÕI của SQL LIKE:
    Hệ thống phải đoán từ khoá — không hiểu ngữ nghĩa.
    """
    STOP_WORDS = {
        "with", "and", "the", "for", "in", "of", "to", "a", "an",
        "is", "are", "that", "this", "have", "has", "been", "will",
        "can", "from", "or", "on", "at", "by", "who", "our"
    }
    words = re.findall(r'[a-zA-Z]+', query.lower())
    keywords = [w for w in words if len(w) >= 3 and w not in STOP_WORDS]
    return keywords


def sql_like_search_cv(query: str, limit: int = 5) -> dict:
    """
    Tìm kiếm CV bằng SQL LIKE.
    
    Chiến lược 2 bước:
    1. AND search: tất cả keywords phải khớp (precision cao, recall thấp)
    2. OR search : ít nhất 1 keyword khớp, sắp xếp theo số keywords khớp
    
    Trả về dict với metadata để phân tích sau.
    """
    keywords = extract_keywords(query)
    start_time = time.perf_counter()
    
    if not keywords:
        return {"results": [], "keywords": [], "strategy": "none",
                "elapsed_ms": 0, "query": query}
    
    # === Bước 1: AND search (tất cả keywords) ===
    and_conditions = " AND ".join(
        f"(cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' "
        f"OR highlights LIKE '%{kw}%' OR keyword LIKE '%{kw}%')"
        for kw in keywords
    )
    # Tính số keyword khớp để dùng làm score tương đối
    match_score_expr = " + ".join(
        f"(CASE WHEN cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' "
        f"OR highlights LIKE '%{kw}%' THEN 1 ELSE 0 END)"
        for kw in keywords
    )
    
    and_sql = f"""
        SELECT id, position, keyword, cv_text, highlights,
               ({match_score_expr}) AS match_score
        FROM cvs
        WHERE {and_conditions}
        ORDER BY match_score DESC
        LIMIT {limit}
    """
    cur.execute(and_sql)
    and_results = [dict(row) for row in cur.fetchall()]
    
    strategy = "AND"
    results = and_results
    
    # === Bước 2: Fallback sang OR nếu AND không có kết quả ===
    if not and_results:
        strategy = "OR (fallback)"
        or_conditions = " OR ".join(
            f"(cv_text LIKE '%{kw}%' OR position LIKE '%{kw}%' "
            f"OR highlights LIKE '%{kw}%' OR keyword LIKE '%{kw}%')"
            for kw in keywords
        )
        or_sql = f"""
            SELECT id, position, keyword, cv_text, highlights,
                   ({match_score_expr}) AS match_score
            FROM cvs
            WHERE {or_conditions}
            ORDER BY match_score DESC
            LIMIT {limit}
        """
        cur.execute(or_sql)
        results = [dict(row) for row in cur.fetchall()]
    
    elapsed_ms = (time.perf_counter() - start_time) * 1000
    
    return {
        "query":      query,
        "keywords":   keywords,
        "strategy":   strategy,
        "results":    results,
        "elapsed_ms": elapsed_ms,
    }


def print_search_results(search_result: dict):
    """In kết quả tìm kiếm theo định dạng dễ đọc"""
    q   = search_result["query"]
    kws = search_result["keywords"]
    stg = search_result["strategy"]
    ms  = search_result["elapsed_ms"]
    res = search_result["results"]
    
    print(f"\n{'='*65}")
    print(f"🔍 Query     : {q}")
    print(f"   Keywords  : {kws}")
    print(f"   Strategy  : SQL LIKE [{stg}]")
    print(f"   Thời gian : {ms:.2f} ms")
    print(f"   Kết quả   : {len(res)} ứng viên")
    print(f"{'='*65}")
    
    if not res:
        print("  ❌ KHÔNG TÌM ĐƯỢC ứng viên nào!")
        print("     → SQL LIKE thất bại hoàn toàn với query ngữ nghĩa này")
        return
    
    table_data = []
    for i, row in enumerate(res, 1):
        matched_kw = row.get("keyword", "")[:40]
        snippet    = row.get("cv_text", "")[:80] + "..."
        score_bar  = "★" * int(row["match_score"]) + "☆" * (len(kws) - int(row["match_score"]))
        table_data.append([
            f"#{i}",
            row["position"][:30],
            f"{row['match_score']}/{len(kws)} {score_bar}",
            matched_kw,
            snippet,
        ])
    
    print(tabulate(
        table_data,
        headers=["#", "Position", "Match Score", "Keyword", "CV Snippet"],
        tablefmt="rounded_outline",
        maxcolwidths=[4, 30, 15, 40, 80]
    ))

print("✅ Hàm SQL LIKE search đã sẵn sàng")

✅ Hàm SQL LIKE search đã sẵn sàng


In [6]:
# ============================================================
# CELL 6: Chạy các test query — CÙNG query như Milvus notebook
# ============================================================
# 5 query được thiết kế để lộ rõ điểm yếu của SQL LIKE:
#   Q1: Từ khoá kỹ thuật rõ ràng  → SQL LIKE có thể xử lý được
#   Q2: Ngữ nghĩa / paraphrase     → SQL LIKE bắt đầu gặp khó
#   Q3: Synonym hoàn toàn          → SQL LIKE thất bại
#   Q4: Mô tả văn hoá công ty      → SQL LIKE không hiểu
#   Q5: Câu hỏi tự nhiên phức tạp  → SQL LIKE sụp đổ

queries = [
    # Q1 — Cùng query chính trong Milvus notebook
    "Python backend developer with Django and REST API experience",

    # Q2 — Paraphrase: không viết "Python" mà mô tả bằng cách khác
    "server-side engineer who builds web APIs and database integrations",

    # Q3 — Hoàn toàn synonym: startup agile → môi trường linh hoạt
    "someone who thrives in fast-paced small teams with rapid iteration",

    # Q4 — Mô tả kỹ năng mềm + văn hoá
    "strong communicator who can lead cross-functional projects",

    # Q5 — Query rất tự nhiên, không có từ khoá kỹ thuật rõ ràng
    "I need a data person who understands both the business side and technical implementation",
]

all_results = []
for q in queries:
    result = sql_like_search_cv(q, limit=5)
    all_results.append(result)
    print_search_results(result)


🔍 Query     : Python backend developer with Django and REST API experience
   Keywords  : ['python', 'backend', 'developer', 'django', 'rest', 'api', 'experience']
   Strategy  : SQL LIKE [OR (fallback)]
   Thời gian : 12.77 ms
   Kết quả   : 5 ứng viên
╭─────┬──────────────┬───────────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────╮
│ #   │ Position     │ Match Score   │ Keyword   │ CV Snippet                                                                      │
├─────┼──────────────┼───────────────┼───────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ #1  │ 1C developer │ 3/7 ★★★☆☆☆☆   │ Flutter   │ 1 am an 1C developer. I deployed an 1C to typographical factory in Ukraine.     │
│     │              │               │           │ Also...                                                                         │
│ #2  │ 1C Developer │ 3/7 ★★★☆☆☆☆   │ Other     │ Perfect knowledge of 1C:Enter

In [7]:
# ============================================================
# CELL 7: Phân tích so sánh SQL LIKE vs Vector Search (Milvus)
# ============================================================
print("\n" + "#"*65)
print("#     PHÂN TÍCH SO SÁNH: SQL LIKE vs MILVUS VECTOR SEARCH     #")
print("#"*65)

# --- Thống kê kết quả SQL LIKE ---
print("\n📊 KẾT QUẢ SQL LIKE theo từng query:")
summary_data = []
empty_count  = 0
fallback_count = 0

for r in all_results:
    n_results = len(r["results"])
    is_empty    = n_results == 0
    is_fallback = "fallback" in r["strategy"]
    if is_empty:    empty_count += 1
    if is_fallback: fallback_count += 1

    avg_score = (
        sum(x["match_score"] for x in r["results"]) / n_results
        if n_results > 0 else 0
    )
    status = "❌ TRỐNG" if is_empty else ("⚠️  Fallback OR" if is_fallback else "✅ AND")
    summary_data.append([
        r["query"][:50],
        str(r["keywords"][:4]),
        status,
        n_results,
        f"{avg_score:.1f}/{len(r['keywords'])}",
        f"{r['elapsed_ms']:.1f} ms",
    ])

print(tabulate(
    summary_data,
    headers=["Query", "Keywords trích xuất", "Strategy", "Kết quả", "Avg Score", "Thời gian"],
    tablefmt="rounded_outline",
    maxcolwidths=[50, 30, 15, 8, 10, 10]
))

# --- Bảng so sánh chi tiết ---
print("\n\n📋 BẢNG SO SÁNH CHI TIẾT:")
comparison = [
    ["Kỹ thuật tìm kiếm",
     "SQL LIKE '%keyword%'",
     "Cosine Similarity (Vector)"],

    ["Hiểu ngữ nghĩa",
     "❌ Không — chỉ so sánh ký tự",
     "✅ Có — hiểu ý nghĩa câu"],

    ["Xử lý synonym",
     "❌ Thất bại\n(startup ≠ fast-paced team)",
     "✅ Thành công\n(hiểu cùng ý nghĩa)"],

    ["Query bằng câu tự nhiên",
     "❌ Phải tách keywords thủ công",
     "✅ Nhập câu nguyên, tìm ngay"],

    ["Kết quả khi không có keyword",
     "❌ Trả về rỗng",
     "✅ Vẫn trả về kết quả gần nhất"],

    ["Sắp xếp kết quả",
     "⚠️  Đếm số keyword khớp (thô)",
     "✅ Score 0-1 chính xác (cosine)"],

    ["Xử lý lỗi chính tả",
     "❌ 'Pythoon' → không tìm được",
     "✅ Vector vẫn gần đúng"],

    ["Multimodal (ảnh + text)",
     "❌ Không hỗ trợ",
     "✅ Kết hợp vector ảnh + text"],

    ["Tốc độ với 1M+ bản ghi",
     "❌ Chậm — Full Table Scan",
     "✅ Nhanh — HNSW/IVF index"],

    ["Độ phức tạp triển khai",
     "✅ Đơn giản — chỉ cần SQL",
     "⚠️  Cần setup Milvus + model"],

    ["Chi phí tính toán",
     "✅ Thấp",
     "⚠️  Cao hơn (embed + vector)"],
]

print(tabulate(
    comparison,
    headers=["Tiêu chí", "SQL LIKE", "Milvus Vector Search"],
    tablefmt="rounded_outline",
    maxcolwidths=[30, 40, 40]
))

# --- Kết luận ---
print(f"""
╔══════════════════════════════════════════════════════════════╗
║                        KẾT LUẬN                             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  SQL LIKE với {empty_count}/5 query KHÔNG tìm được kết quả          ║
║  SQL LIKE với {fallback_count}/5 query phải fallback sang OR (kém chính xác) ║
║                                                              ║
║  Vấn đề cốt lõi của SQL LIKE trong tuyển dụng:              ║
║  → Nhà tuyển dụng viết câu tự nhiên, không phải từ khoá      ║
║  → Ứng viên dùng từ ngữ khác nhau cho cùng kỹ năng           ║
║  → Không thể tìm "ngôn ngữ linh hoạt" từ query "Python"      ║
║                                                              ║
║  ✅ Milvus giải quyết tất cả vấn đề trên bằng vector space   ║
║     — nơi ngữ nghĩa tương tự = khoảng cách vector nhỏ        ║
╚══════════════════════════════════════════════════════════════╝
""")


#################################################################
#     PHÂN TÍCH SO SÁNH: SQL LIKE vs MILVUS VECTOR SEARCH     #
#################################################################

📊 KẾT QUẢ SQL LIKE theo từng query:
╭────────────────────────────────────────────────────┬────────────────────────────────┬─────────────────┬───────────┬─────────────┬─────────────╮
│ Query                                              │ Keywords trích xuất            │ Strategy        │   Kết quả │ Avg Score   │ Thời gian   │
├────────────────────────────────────────────────────┼────────────────────────────────┼─────────────────┼───────────┼─────────────┼─────────────┤
│ Python backend developer with Django and REST API  │ ['python', 'backend',          │ ⚠️  Fallback OR │         5 │ 3.0/7       │ 12.8 ms     │
│                                                    │ 'developer', 'django']         │                 │           │             │             │
│ server-side engineer who builds we